# 03｜站點每小時觀測借車量基準
用 1–24 日有借車紀錄的站點建立小時網格，25–31 日評估。網格中的零表示檔案內沒有可歸站的借車紀錄，站點營運覆蓋仍待核對。

In [1]:
from zipfile import ZipFile
import pandas as pd
start, split, end = map(pd.Timestamp, ['2025-01-01', '2025-01-25', '2025-02-01'])
with ZipFile('../data/raw/202501-capitalbikeshare-tripdata.zip') as z:
    with z.open('202501-capitalbikeshare-tripdata.csv') as f:
        rides = pd.read_csv(f, usecols=['started_at', 'start_station_id'], dtype='string')
hour = pd.to_datetime(rides.started_at, format='mixed', errors='coerce').dt.floor('h')
station = rides.start_station_id.str.strip().replace('', pd.NA)
events = pd.DataFrame({'station': station, 'hour': hour})
events = events.loc[hour.ge(start) & hour.lt(end) & station.notna()]
stations = sorted(events.loc[events.hour.lt(split), 'station'].unique())
cohort = events[events.station.isin(stations)]
grid = pd.MultiIndex.from_product(
    [stations, pd.date_range(start, end, freq='h', inclusive='left')],
    names=['station', 'hour'])
panel = cohort.groupby(['station', 'hour']).size().reindex(grid, fill_value=0).rename('actual').to_frame()
assert panel.actual.sum() == len(cohort)
print('Stations:', len(stations))
print('Excluded test events from unseen stations:', len(events) - len(cohort))

Stations: 762
Excluded test events from unseen stations: 30


逐小時預測。每個預測只使用同站已結束的上一小時。MAE 衡量每個站點小時平均差幾次借車。

In [2]:
panel['previous_hour'] = panel.groupby(level='station').actual.shift(1)
test = panel.loc[panel.index.get_level_values('hour') >= split].copy()
assert test.previous_hour.notna().all()
test['always_zero'] = 0
print('Test station-hours:', len(test))
print(f'Observed-zero share: {test.actual.eq(0).mean():.2%}')
for name in ['always_zero', 'previous_hour']:
    errors = (test[name] - test.actual).abs()
    print(f'{name}: MAE={errors.mean():.4f}, positive-hour MAE={errors[test.actual.gt(0)].mean():.4f}')

Test station-hours: 128016
Observed-zero share: 77.52%
always_zero: MAE=0.5057, positive-hour MAE=2.2498
previous_hour: MAE=0.4950, positive-hour MAE=1.5790


限制：只有一個月，站點未知事件未分配，時區與營運狀態尚未核對。逐小時更新使用最新觀測，評估不適用於一次預測整週的情境。